> Bu, Tom Augspurger'ın CC-BY-4.0 lisanslı 'Effective Pandas' eserinin gönüllü Türkçe çevirisidir. Orijinal: https://github.com/TomAugspurger/effective-pandas

# Effective Pandas

## Giriş (Introduction)

Bu seri, Python programlama dili için bir veri analizi kütüphanesi olan [pandas](http://pandas.pydata.org)'ı etkili bir şekilde nasıl kullanacağınızı ele alır.
Orta seviyeye yöneliktir: pandas ile biraz deneyimi olan ama kendini geliştirmek isteyen kişilere hitap eder.

## Önceki Çalışmalar (Prior Art)

pandas öğrenmek için pek çok harika kaynak var; bu yazı onlardan biri değil.
Yeni başlayanlar için, özellikle SQL'e aşinaysalar, genellikle [Greg Reda'nın](https://twitter.com/gjreda) [3 bölümlük girişini](http://gregreda.com/2013/10/26/intro-to-pandas-data-structures/) öneririm. Elbette pandas'ın kendi [dokümantasyonu](http://pandas.pydata.org/) da var. Video formatını tercih edenler için PyData Seattle'da giriş niteliğinde bir [konuşma](https://www.youtube.com/watch?v=otCriSKVV_8) yaptım. Wes McKinney'nin [Python for Data Analysis](http://shop.oreilly.com/product/0636920023784.do) kitabı hâlâ başvurulacak kaynak (aynı zamanda NumPy'a da gerçekten iyi bir giriş niteliğinde). Jake VanderPlas'ın erken sürümde olan [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) kitabı da harika.
Kevin Markham'ın pandas öğrenmeye yeni başlayanlar için bir [video serisi](http://www.dataschool.io/easier-data-analysis-with-pandas/) var.

Tüm bu kaynaklara rağmen (ve değinmeden geçtiğim daha pek çoğuna rağmen) neden bir tane daha yazayım? Artık azalan verim yasası devreye girmiş olmalı.
Yine de, güncel (Mart 2016 itibarıyla) ve deyimsel (idiomatic) pandas koduna -yani *pandorable* olan koda- ağırlık veren bir rehbere yer olduğunu düşündüm. <!-- TR-NOT: 'pandorable', yazarın 'pythonic' kelimesine gönderme yaparak türettiği bir söz oyunu; sözlükte karşılığı olmadığı için İngilizcesi korunmuştur. -->
Bu seri muhtemelen python'a ya da NumPy ve pandas'a tamamen yeni başlayanlar için uygun olmayacak.
Şans eseri, bu ilk yazı görece giriş niteliğinde konuları kapsıyor;
bu yüzden bağlantı verilen kaynaklardan bazılarını okuyup geri dönebilir, ya da sorularınız varsa [bana ulaşabilirsiniz](https://twitter.com/tomaugspurger).

## Veriyi Edinme (Get the Data)

BTS'den alınan [uçuş gecikmesi verileriyle](http://www.transtats.bts.gov/databases.asp?Mode_ID=1&Mode_Desc=Aviation&Subject_ID2=0) çalışacağız (R kullanıcıları benzer veriler için Hadley'nin [NYCFlights13](https://github.com/hadley/nycflights13) veri kümesini kurabilir.



In [1]:
import os
import zipfile

import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if int(os.environ.get("MODERN_PANDAS_EPUB", 0)):
    import prep

In [2]:
import requests

headers = {
    'Referer': 'https://www.transtats.bts.gov/DL_SelectFields.asp?Table_ID=236&DB_Short_Name=On-Time',
    'Origin': 'https://www.transtats.bts.gov',
    'Content-Type': 'application/x-www-form-urlencoded',
}

params = (
    ('Table_ID', '236'),
    ('Has_Group', '3'),
    ('Is_Zipped', '0'),
)

with open('modern-1-url.txt', encoding='utf-8') as f:
    data = f.read().strip()

os.makedirs('data', exist_ok=True)
dest = "data/flights.csv.zip"

if not os.path.exists(dest):
    r = requests.post('https://www.transtats.bts.gov/DownLoad_Table.asp',
                      headers=headers, params=params, data=data, stream=True)

    with open("data/flights.csv.zip", 'wb') as f:
        for chunk in r.iter_content(chunk_size=102400): 
            if chunk:
                f.write(chunk)

Bu indirme işlemi bir ZIP dosyası döndürdü.
Tek bir CSV içeren ZIP arşivlerini otomatik olarak açmak için açık bir [Pull Request](https://github.com/pydata/pandas/pull/12175) bulunuyor,
ama şimdilik bunu kendimiz çıkarıp ardından okumamız gerekiyor.

In [3]:
zf = zipfile.ZipFile("data/flights.csv.zip")
fp = zf.extract(zf.filelist[0].filename, path='data/')
df = pd.read_csv(fp, parse_dates=["FL_DATE"]).rename(columns=str.lower)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450017 entries, 0 to 450016
Data columns (total 33 columns):
fl_date                  450017 non-null datetime64[ns]
unique_carrier           450017 non-null object
airline_id               450017 non-null int64
tail_num                 449378 non-null object
fl_num                   450017 non-null int64
origin_airport_id        450017 non-null int64
origin_airport_seq_id    450017 non-null int64
origin_city_market_id    450017 non-null int64
origin                   450017 non-null object
origin_city_name         450017 non-null object
dest_airport_id          450017 non-null int64
dest_airport_seq_id      450017 non-null int64
dest_city_market_id      450017 non-null int64
dest                     450017 non-null object
dest_city_name           450017 non-null object
crs_dep_time             450017 non-null int64
dep_time                 441476 non-null float64
dep_delay                441476 non-null float64
taxi_out                

## İndeksleme (Indexing)

Ya da, *açık olmak, kapalı olmaktan iyidir*.
Saydığım kadarıyla, [Stackoverflow](http://stackoverflow.com/questions/tagged/pandas?sort=votes&pageSize=15)'daki en çok oy alan ilk 15 pandas sorusunun 7'si indeksleme hakkında. Başlamak için bundan daha iyi bir yer olamaz.

İndeksleme derken, bir DataFrame ya da Series'in alt kümelerinin seçilmesini kastediyoruz.
`DataFrame`'ler (ve daha az ölçüde `Series`'ler) zorlu bir dizi meydan okuma sunar:

- Listeler gibi, konuma göre indeksleyebilirsiniz.
- Sözlükler gibi, etikete (label) göre indeksleyebilirsiniz.
- NumPy dizileri gibi, boolean maskelere göre indeksleyebilirsiniz.
- Bu indeksleyicilerin herhangi biri skaler bir indeks, bir dizi ya da bir `slice` olabilir.
- Bunların herhangi biri, bir DataFrame'in indeksi (satır etiketleri) ya da sütunları üzerinde çalışabilmelidir.
- Ve bunların herhangi biri, hiyerarşik indeksler üzerinde de çalışabilmelidir.

pandas'ın indekslemesindeki karmaşıklık, genel olarak pandas API'sinin karmaşıklığının küçük bir örneğidir.
Bu karmaşıklığın bir nedeni var (çoğunun en azından), ama bu, öğrenirken *pek de* teselli olmuyor.
Yine de, tüm bu indeksleme yöntemleri, kütüphaneye dahil edilmelerini haklı çıkaracak kadar gerçekten kullanışlıdır.

## Dilimleme (Slicing)

Ya da, *açık olmak, kapalı olmaktan iyidir*.

Saydığım kadarıyla, [Stackoverflow](http://stackoverflow.com/questions/tagged/pandas?sort=votes&pageSize=15)'daki en çok oy alan ilk 15 pandas sorusunun 7'si dilimleme hakkında. Başlamak için bundan daha iyi bir yer olamaz.

Kısa bir tarihsel not: Yıllarca satır ve/veya sütun seçimi için tercih edilen yöntem `.ix` idi.

In [5]:
df.ix[10:15, ['fl_date', 'tail_num']]

,fl_date,tail_num
10,2014-01-01,N3LGAA
11,2014-01-01,N368AA
12,2014-01-01,N3DDAA
13,2014-01-01,N332AA
14,2014-01-01,N327AA
15,2014-01-01,N3LBAA


Ancak bu basit görünen küçük işlem bazı karmaşıklıkları gizliyor. Ya varsayılan `range(n)` indeksimiz yerine, aşağıdaki gibi bir tam sayı indeksimiz olsaydı

In [6]:
first = df.groupby('airline_id')[['fl_date', 'unique_carrier']].first()
first.head()

,fl_date,unique_carrier
airline_id,,
19393,2014-01-01,WN
19690,2014-01-01,HA
19790,2014-01-01,DL
19805,2014-01-01,AA
19930,2014-01-01,AS


Yukarıdaki dilimimizi `.ix`'e verdiğimizde ne döneceğini önceden tahmin edebilir misiniz?

In [7]:
first.ix[10:15, ['fl_date', 'tail_num']]

,fl_date,tail_num
airline_id,,


Sürpriz, boş bir DataFrame! Bu da veri analizinde nadiren iyi bir şey olur. Ne oldu?

Tam sayı bir indeksimiz vardı, dolayısıyla `.ix` çağrısı etikete dayalı (label-based) modunu kullandı. 10 ile 15 arasında (dahil) tam sayı *etiketler* arıyordu. Hiçbirini bulamadı. Bir aralığı dilimlediğimiz için bir KeyError fırlatmak yerine boş bir DataFrame döndürdü.

Bunun aksine, tam sayı yerine bir string indeksimiz olduğunu varsayalım.

In [8]:
first = df.groupby('unique_carrier').first()
first.ix[10:15, ['fl_date', 'tail_num']]

,fl_date,tail_num
unique_carrier,,
UA,2014-01-01,N14214
US,2014-01-01,N650AW
VX,2014-01-01,N637VA
WN,2014-01-01,N412WN



Ve yine çalışıyor! Artık bir string indeksimiz olduğu için `.ix` konumsal (positional) modunu kullandı. 10-15 arasındaki *satırları* aradı (sağ taraf hariç).

Ancak dilimlemenin sonucunu önceden güvenilir bir şekilde tahmin edemezsiniz. Kodun *okuyucusuna* (muhtemelen gelecekteki siz) düşen şey, veri tiplerini (dtype) bilerek `.ix`'in etiket tabanlı indekslemeyi mi (boş DataFrame döndüren) yoksa konumsal indekslemeyi mi (son örnekteki gibi) kullanacağını kestirebilmektir.
Genel olarak, `.ix`'in tam sayı Index'lerde etikete dayalı indekslemeye, tam sayı olmayanlarda ise konuma dayalı indekslemeye yönlendirmesi gibi davranışı veriye bağlı olan metotları doğru kullanmak zordur. pandas'ta bunları ortadan kaldırmaya çalışıyoruz.

pandas 0.12'den bu yana, bu görevler iki metoda net bir şekilde ayrılmıştır:

1. Etiket tabanlı indeksleme için `.loc`
2. Konumsal indeksleme için `.iloc`

In [9]:
first.loc[['AA', 'AS', 'DL'], ['fl_date', 'tail_num']]

,fl_date,tail_num
unique_carrier,,
AA,2014-01-01,N338AA
AS,2014-01-01,N524AS
DL,2014-01-01,N911DL


In [10]:
first.iloc[[0, 1, 3], [0, 1]]

,fl_date,airline_id
unique_carrier,,
AA,2014-01-01,19805
AS,2014-01-01,19930
DL,2014-01-01,19790


`.ix` hâlâ kullanılabilir durumda ve yakın zamanda kullanımdan kaldırılması (deprecated) planlanmıyor. Zaman zaman işe yarıyor. Ama `.ix`'i alışkanlıktan kullanıyorsanız, ya da daha iyisini bilmiyorsanız, `.loc` ve `.iloc`'a bir şans verin. Meraklı okuyucular için, Joris Van den Bossche (pandas'ın çekirdek geliştiricilerinden biri) pandas `__getitem__` API'sine dair [harika bir özet derledi](https://github.com/pydata/pandas/issues/9595).
Bu serideki ileriki bir yazı, Indeks'leri etkili kullanma konusuna daha ayrıntılı girecek;
bunlar kendi başlarına da kullanışlı nesnelerdir, ama şimdilik yakından ilişkili bir konuya geçelim.

## SettingWithCopy

pandas eskiden, atamaların çalışmıyormuş gibi göründüğüne dair *bir sürü* soru alırdı. Temsili bir örnek olarak [şu StackOverflow](http://stackoverflow.com/q/16553298/1889400) sorusunu ele alacağız.

In [11]:
f = pd.DataFrame({'a':[1,2,3,4,5], 'b':[10,20,30,40,50]})
f

,a,b
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


Kullanıcı, `a` değeri 3 veya daha küçük olan satırlardaki `b` değerlerini alıp bunları `b / 10`'a eşitlemek istiyordu.
Bu satırları seçmek için boolean indeksleme kullanacağız: `f['a'] <= 3`,

In [12]:
# ignore the context manager for now
with pd.option_context('mode.chained_assignment', None):
    f[f['a'] <= 3]['b'] = f[f['a'] <= 3 ]['b'] / 10
f

,a,b
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


Ve hiçbir şey olmadı. Aslında bir şey oldu, ama kimse buna tanık olmadı. Hiçbir referansı olmayan bir nesne değiştirilirse, bunun bir sesi çıkar mı?

Yukarıda context manager ile sessize aldığım uyarı, oldukça faydalı [bir açıklamaya](http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy) bağlanıyor. Burada önemli noktaları özetleyeceğim.

`f`'nin güncellenme "başarısızlığı", *zincirleme indeksleme (chained indexing)* denen ve kaçınılması gereken bir uygulamadan kaynaklanır.
"Zincirleme" ifadesi, tek bir indeksleme işlemi yerine, birbiri ardına birden fazla kez indeksleme yapılmasından gelir.
Yukarıda, eşittir işaretinin solunda iki işlem vardı: biri `__getitem__`, diğeri `__setitem__` (python'da köşeli parantezler, atama söz konusuysa `__getitem__` ya da `__setitem__` için sözdizimsel şekerdir - syntactic sugar). Yani `f[f['a'] <= 3]['b']` şu hale gelir:

1. `getitem`: `f[f['a'] <= 3]`
2. `setitem`: `_['b'] = ...`  # 1. adımın sonucunu temsil etmek için `_` kullanılıyor.

Genel olarak pandas, ilk `__getitem__`'in altta yatan verinin bir view'ını mı yoksa bir kopyasını mı döndürdüğünü garanti edemez.
Değişiklikler, yukarıda `_` olarak adlandırdığım şeye, yani 1. adımdaki `__getitem__`'in sonucuna *uygulanacaktır*.
Ama `_`'nin orijinal `f` ile aynı belleği paylaştığını bilmiyoruz.
Dolayısıyla `_` üzerinde yapılan değişikliklerin `f`'ye yansıyıp yansımayacağından emin olamayız.

Doğru şekilde yapmak isterseniz şöyle yazmanız gerekir:

In [13]:
f.loc[f['a'] <= 3, 'b'] = f.loc[f['a'] <= 3, 'b'] / 10
f

,a,b
0,1,1.0
1,2,2.0
2,3,3.0
3,4,40.0
4,5,50.0


Artık her şey tek bir `__setitem__` çağrısında gerçekleşiyor ve pandas atamanın doğru şekilde yapılmasını garanti edebiliyor.

Kabaca kural şu: art arda köşeli parantezler, `][`, gördüğünüz her an başınız derde girebilir. Bunun yerine bir `.loc[..., ...]` kullanın, işiniz hallolsun.

Bir diğer öneri de, SettingWithCopy uyarısının *atama* yapıldığı anda tetiklenmesidir.
Olası kopyalama, kodunuzda çok daha önce gerçekleşmiş olabilir.

## Çok Boyutlu İndeksleme (Multidimensional Indexing)

MultiIndex'ler muhtemelen pandas'ın en sevdiğim özelliği.
Daha yüksek boyutlu veri kümelerini, beynimin bazen kaldırabildiği tanıdık iki boyutlu bir tabloda temsil etmenizi sağlarlar.
MultiIndex'in her ek seviyesi başka bir boyutu temsil eder.
Bunun bedeli ise biraz daha zor bir etiket indekslemesidir.

pandas'a gönderdiğim ilk hata raporu, [Kasım 2012](https://github.com/pydata/pandas/issues/2207)'ye kadar uzanıyor;
bir MultiIndex içine indeksleme yapmakla ilgiliydi.
Bunu şimdi anlatıyorum çünkü aldığım sonucun gerçekten bir hata mı olduğunu anlayamamıştım.
Ayrıca, o hata raporundan bir alıntı:

> Bu gerçekten bir hata değilse özür dilerim. Python'da hâlâ çok yeniyim. Teşekkürler!

Çok tatlı.

Bu işlem, MultiIndex'in rastgele seviyelerini dilimlemenizi sağlayan 2014'teki [şu](http://pandas.pydata.org/pandas-docs/version/0.18.0/whatsnew.html#multiindexing-using-slicers) eklentiyle çok daha kolay hale geldi.
Üzerinde çalışmak için bir MultiIndex'li DataFrame oluşturalım.

In [14]:
hdf = df.set_index(['unique_carrier', 'origin', 'dest', 'tail_num', 'fl_date']).sort_index()
hdf[hdf.columns[:4]].head()

airline_id  fl_num  \
unique_carrier origin dest tail_num fl_date                          
AA             ABQ    DFW  N200AA   2014-01-06       19805    1662   
                                    2014-01-27       19805    1090   
                           N202AA   2014-01-27       19805    1332   
                           N426AA   2014-01-09       19805    1662   
                                    2014-01-15       19805    1467   

                                                origin_airport_id  \
unique_carrier origin dest tail_num fl_date                         
AA             ABQ    DFW  N200AA   2014-01-06              10140   
                                    2014-01-27              10140   
                           N202AA   2014-01-27              10140   
                           N426AA   2014-01-09              10140   
                                    2014-01-15              10140   

                                                origin_airport_seq_id  
unique_carrier origin dest tail_num fl_date                            
AA             ABQ    DFW  N200AA   2014-01-06                1014002  
                                    2014-01-27                1014002  
                           N202AA   2014-01-27                1014002  
                           N426AA   2014-01-09                1014002  
                                    2014-01-15                1014002

Terminolojiyi netleştirmek gerekirse, bir MultiIndex'in *seviyeleri (levels)*,
eski sütun adlarıdır (`unique_carrier`, `origin`...).
Etiketler (labels) ise bir seviyedeki gerçek değerlerdir (`'AA'`, `'ABQ'`, ...).
Seviyelere isimle ya da konumla başvurulabilir; 0, en dıştaki seviyeyi temsil eder.

En dıştaki indeks seviyesini dilimlemek oldukça kolaydır, sadece alışıldık `.loc[row_indexer, column_indexer]` kullanırız. Taşıyıcının American Airlines, Delta ya da US Airways olduğu satırlar için `dep_time` ve `dep_delay` sütunlarını seçeceğiz.

In [15]:
hdf.loc[['AA', 'DL', 'US'], ['dep_time', 'dep_delay']]

dep_time  dep_delay
unique_carrier origin dest tail_num fl_date                        
AA             ABQ    DFW  N200AA   2014-01-06    1246.0       71.0
                                    2014-01-27     605.0        0.0
                           N202AA   2014-01-27     822.0      -13.0
                           N426AA   2014-01-09    1135.0        0.0
                                    2014-01-15    1022.0       -8.0
...                                                  ...        ...
US             TUS    PHX  N824AW   2014-01-16    1900.0      -10.0
                                    2014-01-20    1903.0       -7.0
                           N836AW   2014-01-08    1928.0       18.0
                                    2014-01-29    1908.0       -2.0
                           N837AW   2014-01-10    1902.0       -8.0

[139194 rows x 2 columns]

Şimdiye kadar her şey yolunda. Peki ya kalkış noktası Chicago O'Hare (`ORD`) ya da Des Moines Uluslararası Havalimanı (DSM) olan satırları seçmek isteseydiniz?
`.loc`, `[row_indexer, column_indexer]` istiyor, o yüzden satır indeksleyicimizin iki öğesini (taşıyıcı listesi ve kalkış noktası listesi) tek bir birim haline getirmek için bir tuple içine alalım:

In [16]:
hdf.loc[(['AA', 'DL', 'US'], ['ORD', 'DSM']), ['dep_time', 'dep_delay']]

dep_time  dep_delay
unique_carrier origin dest tail_num fl_date                        
AA             DSM    DFW  N200AA   2014-01-12     603.0       -7.0
                                    2014-01-17     751.0      101.0
                           N424AA   2014-01-10    1759.0       -1.0
                                    2014-01-15    1818.0       18.0
                           N426AA   2014-01-07    1835.0       35.0
...                                                  ...        ...
US             ORD    PHX  N806AW   2014-01-26    1406.0       -4.0
                           N830AW   2014-01-28    1401.0       -9.0
                           N833AW   2014-01-10    1500.0       50.0
                           N837AW   2014-01-19    1408.0       -2.0
                           N839AW   2014-01-14    1406.0       -4.0

[5205 rows x 2 columns]

Şimdi de sadece bu taşıyıcılardan değil, ORD ya da DSM'den kalkan herhangi bir uçuşu bulmayı deneyin.
Bu eskiden can sıkıcı bir işti.
`.xs` metoduna başvurmanız, ya da `df.index.get_level_values(0)`'ı kullanıp istediğiniz indeksleyicilerle zip'lemeniz, ya da belki indeksi sıfırlayıp bir boolean maske uygulayıp indeksi yeniden ayarlamanız gerekebilirdi... çok yorucu.

Ama artık bir `IndexSlice` kullanabilirsiniz.

In [17]:
hdf.loc[pd.IndexSlice[:, ['ORD', 'DSM']], ['dep_time', 'dep_delay']]

dep_time  dep_delay
unique_carrier origin dest tail_num fl_date                        
AA             DSM    DFW  N200AA   2014-01-12     603.0       -7.0
                                    2014-01-17     751.0      101.0
                           N424AA   2014-01-10    1759.0       -1.0
                                    2014-01-15    1818.0       18.0
                           N426AA   2014-01-07    1835.0       35.0
...                                                  ...        ...
WN             DSM    MDW  N941WN   2014-01-17    1759.0       14.0
                           N943WN   2014-01-10    2229.0      284.0
                           N963WN   2014-01-22     656.0       -4.0
                           N967WN   2014-01-30     654.0       -6.0
                           N969WN   2014-01-19    1747.0        2.0

[22380 rows x 2 columns]

`:` işareti, bu seviyedeki her etiketin dahil edileceğini belirtir.
`IndexSlice` nesnesi, her seviyeyi dilimlemek için gereken asıl python `slice` nesnesinin sadece bir şekeridir (syntactic sugar). <!-- TR-NOT: Orijinal cümledeki ifade biraz belirsizdi; 'her seviyeyi dilimlemek için gereken slice nesnesinin şekeri' anlamıyla çevrildi. -->

In [18]:
pd.IndexSlice[:, ['ORD', 'DSM']]

(slice(None, None, None), ['ORD', 'DSM'])

`IndexSlice` kullanıyoruz çünkü `hdf.loc[(:, ['ORD', 'DSM'])]` geçerli bir python sözdizimi değil.
Artık gönlümüzce dilimleme yapabiliriz; Ocak ayının ilk yarısında O'Hare'den Des Moines'e tüm uçuşlar mı? Tabii, neden olmasın?

In [19]:
hdf.loc[pd.IndexSlice[:, 'ORD', 'DSM', :, '2014-01-01':'2014-01-15'],
        ['dep_time', 'dep_delay', 'arr_time', 'arr_delay']]

dep_time  dep_delay  arr_time  \
unique_carrier origin dest tail_num fl_date                                     
EV             ORD    DSM  NaN      2014-01-07       NaN        NaN       NaN   
                           N11121   2014-01-05       NaN        NaN       NaN   
                           N11181   2014-01-12    1514.0        6.0    1625.0   
                           N11536   2014-01-10    1723.0        4.0    1853.0   
                           N11539   2014-01-01    1127.0      127.0    1304.0   
...                                                  ...        ...       ...   
UA             ORD    DSM  N24212   2014-01-09    2023.0        8.0    2158.0   
                           N73256   2014-01-15    2019.0        4.0    2127.0   
                           N78285   2014-01-07    2020.0        5.0    2136.0   
                                    2014-01-13    2014.0       -1.0    2114.0   
                           N841UA   2014-01-11    1825.0       20.0    1939.0   

                                                arr_delay  
unique_carrier origin dest tail_num fl_date                
EV             ORD    DSM  NaN      2014-01-07        NaN  
                           N11121   2014-01-05        NaN  
                           N11181   2014-01-12       -2.0  
                           N11536   2014-01-10       19.0  
                           N11539   2014-01-01      149.0  
...                                                   ...  
UA             ORD    DSM  N24212   2014-01-09       34.0  
                           N73256   2014-01-15        3.0  
                           N78285   2014-01-07       12.0  
                                    2014-01-13      -10.0  
                           N841UA   2014-01-11       19.0  

[153 rows x 4 columns]

Indeks'lerle (MultiIndex'ler dahil) çalışma konusuna ileriki bir yazıda daha ayrıntılı değineceğiz. Kanıtlanmamış bir tezim var: `IndexSlice` yeterince kullanılmadığı için Indeks'ler de yeterince kullanılmıyor, bu da insanların onların olduğundan daha hantal olduğunu düşünmesine yol açıyor. Ama birinci bölümü burada kapatalım.

## Toparlama (WrapUp)

Bu ilk yazı, pandas'ın merkezinde yer alan bir konu olan İndeksleme'yi ele aldı.
DataFrame'in sağladığı güç, bazı kaçınılmaz karmaşıklıklarla birlikte gelir.
En iyi uygulamalar (`.loc` ve `.iloc` kullanmak) size pek çok baş ağrısından tasarruf sağlayacaktır.
Ardından, sıkça yanlış anlaşılan birkaç alt konuyu, kopya üzerinde değer atamayı (setting with copy) ve Hiyerarşik İndeksleme'yi ele aldık. <!-- TR-NOT: 'setting with copy' sözlükte yok; pandas'taki SettingWithCopyWarning kavramına atıfla 'kopya üzerinde değer atama' şeklinde çevrildi. -->